# 08 - Regression discontinuity

A threshold rule that decides who gets treated is, locally, almost an
experiment. Units just below the cutoff and units just above it are similar in
everything except treatment — so comparing them isolates the effect, without
requiring that we measured every confounder.

The price is that the answer is local. It applies at the cutoff, and nowhere
else.

## Causal question

A scholarship is awarded to every applicant whose eligibility score reaches a
threshold. Among applicants at that threshold, does receiving the scholarship
change the outcome?

Note what is *not* being asked. This design cannot say what the scholarship
would do for an applicant scoring far below the cutoff.

## Data and design

- **Unit of analysis:** one applicant.
- **Running variable:** `running`, the eligibility score, centred so the cutoff
  sits at zero.
- **Treatment:** `treatment`, deterministic in the running variable — every unit
  at or above the cutoff is treated, and no unit below it is. This is a *sharp*
  design.
- **Outcome:** `outcome`, measured after treatment.

The generator builds the data with a known discontinuity, so the estimate can be
checked against a value we already know.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd

from causal_inference_lab.data_generators import make_sharp_rdd_data
from causal_inference_lab.rdd import local_linear_rdd, rdd_bandwidth_sensitivity

dataset = make_sharp_rdd_data(n=5_000, cutoff=0.0, seed=21)
data = dataset.data

print(f"observations:     {len(data):,}")
print(f"treated:          {int(data['treatment'].sum()):,}")
print(f"true effect:      {dataset.true_ate:.3f}")
print()
print("treatment is a deterministic function of the running variable:")
print(pd.crosstab(data["running"] >= 0.0, data["treatment"]))

observations:     5,000
treated:          2,530
true effect:      2.500

treatment is a deterministic function of the running variable:
treatment     0     1
running              
False      2470     0
True          0  2530


**Interpretation.** The crosstab has zeros off the diagonal: every unit above
the cutoff is treated and every unit below is not. That is what makes the design
sharp, and it is also why no covariate adjustment can help — at any given score
there is only one treatment status, so there is nothing to compare within.

## Estimand

The **effect at the cutoff**: the difference in expected outcomes for units
whose running variable sits exactly at the threshold.

This is not the ATE. It is a local quantity, and treating it as an average
effect for the whole population is the most common way to misreport an RDD.

## Identification assumptions

1. **Continuity.** Expected potential outcomes are continuous in the running
   variable at the cutoff. Any jump in the outcome is caused by treatment, not
   by something else that also changes at the threshold.
2. **No manipulation.** Applicants cannot precisely control their score to land
   just above the cutoff. If they can, those who succeed differ from those who
   do not, and the comparison breaks.
3. **No other policy at the same threshold.** If a second programme uses the
   same cutoff, the estimate captures both.

Continuity is untestable. Manipulation leaves a fingerprint — a pile-up of
observations on the treated side — which we check below.

## Estimation

Local linear regression fits separate lines either side of the cutoff using only
observations inside a bandwidth, and reports the gap between them at the
threshold. Narrow bandwidths reduce bias and raise variance; wide ones do the
reverse.

In [2]:
estimate = local_linear_rdd(data, cutoff=0.0, bandwidth=1.0)

print(f"RDD estimate at cutoff: {estimate.estimate:.3f}")
print(f"standard error:         {estimate.standard_error:.3f}")
print(f"bandwidth:              {estimate.bandwidth}")
print(f"true effect:            {dataset.true_ate:.3f}")

window = data.loc[data["running"].abs() <= estimate.bandwidth]
print(f"\nobservations used:      {len(window):,} of {len(data):,}")

RDD estimate at cutoff: 2.434
standard error:         0.096
bandwidth:              1.0
true effect:            2.500

observations used:      1,642 of 5,000


**Interpretation.** The estimate recovers the known effect closely. Note how
much data the design discards: only observations inside the bandwidth
contribute. An RDD buys credibility with sample size, which is why these
estimates carry wider intervals than a covariate-adjusted estimator on the same
data would.

## Diagnostics

Two checks matter here, and neither is optional.

**Bandwidth sensitivity.** If the estimate swings with the bandwidth, it is
being driven by functional-form choices rather than by a discontinuity.

**Manipulation.** If applicants sorted themselves across the threshold, the
density of the running variable jumps at the cutoff.

In [3]:
sensitivity = rdd_bandwidth_sensitivity(
    data,
    bandwidth_grid=[0.25, 0.5, 1.0, 1.5, 2.0],
    cutoff=0.0,
)
print(sensitivity.to_string(index=False))

spread = sensitivity["estimate"].max() - sensitivity["estimate"].min()
print(f"\nspread across bandwidths: {spread:.3f}")

 bandwidth  estimate  standard_error
      0.25  2.130054        0.189938
      0.50  2.413548        0.136717
      1.00  2.434216        0.096237
      1.50  2.556814        0.079532
      2.00  2.802337        0.069386

spread across bandwidths: 0.672


**Interpretation.** The estimate is not constant: it climbs steadily from about
2.1 at the narrowest bandwidth to about 2.8 at the widest, a spread of roughly a
quarter of the effect itself. That drift is the bias–variance trade-off made
visible. Widening the window pulls in observations further from the cutoff,
where the linear approximation to the outcome curve is worse, and the estimate
absorbs that curvature as if it were treatment effect. Meanwhile the standard
error falls monotonically, because each widening adds data.

The narrow bandwidths bracket the true effect and the widest is visibly biased
away from it. The lesson is not that this design failed — it is that reporting a
single bandwidth without this table would hide a defensible-looking result whose
value depends on a choice the analyst made.

Now the manipulation check: counting observations in narrow bins either side of
the cutoff. Under no manipulation the density should pass through the threshold
smoothly.

In [4]:
edges = np.round(np.arange(-0.5, 0.55, 0.1), 2)
counts = pd.cut(data["running"], bins=edges).value_counts().sort_index()
print(counts.to_string())

just_below = int(data["running"].between(-0.1, 0.0, inclusive="left").sum())
just_above = int(data["running"].between(0.0, 0.1, inclusive="left").sum())
ratio = just_above / just_below

# Counts in a bin are approximately Poisson, so the standard deviation of a
# count near n is about sqrt(n). Judge the gap against that, not against zero.
noise = np.sqrt(just_below + just_above)
print(f"\njust below cutoff: {just_below}")
print(f"just above cutoff: {just_above}")
print(f"ratio:             {ratio:.2f}")
print(f"difference:        {just_above - just_below} ({abs(just_above - just_below) / noise:.1f} sd)")

running
(-0.5, -0.4]    78
(-0.4, -0.3]    80
(-0.3, -0.2]    78
(-0.2, -0.1]    88
(-0.1, -0.0]    93
(-0.0, 0.1]     76
(0.1, 0.2]      92
(0.2, 0.3]      81
(0.3, 0.4]      84
(0.4, 0.5]      74

just below cutoff: 93
just above cutoff: 76
ratio:             0.82
difference:        -17 (1.3 sd)


**Interpretation.** The counts are 93 below and 76 above — a ratio of 0.82, not
1.0. Taken at face value that looks like *fewer* units on the treated side,
which is the opposite of what manipulation produces: applicants who can game a
threshold pile up just above it, not below.

Judged against sampling noise the gap is about 1.3 standard deviations, well
inside what bins of this size produce by chance. So this is consistent with no
manipulation rather than evidence of it. Note the asymmetry in what this check
can do: a clear pile-up on the treated side would be strong evidence *against*
the design, but a clean histogram is only weak evidence *for* it. A formal
McCrary density test is the appropriate tool when the design is load-bearing.

## Uncertainty

The standard error from the local linear fit already reflects the sampling
variability of the units inside the bandwidth. Converting it to an interval
makes the precision explicit.

In [5]:
lower = estimate.estimate - 1.96 * estimate.standard_error
upper = estimate.estimate + 1.96 * estimate.standard_error

print(f"estimate:        {estimate.estimate:.3f}")
print(f"95% interval:    [{lower:.3f}, {upper:.3f}]")
print(f"true effect:     {dataset.true_ate:.3f}")
print(f"covers truth:    {lower <= dataset.true_ate <= upper}")

estimate:        2.434
95% interval:    [2.246, 2.623]
true effect:     2.500
covers truth:    True


**Interpretation.** The interval covers the known effect. It is also noticeably
wide relative to the effect, because it rests on the subset of observations near
the cutoff rather than on all 5,000 — the honest cost of a local design.

## Limitations

- **The estimate is local.** It applies at the cutoff. Extrapolating it to
  applicants far from the threshold is unsupported by this design.
- **Continuity is assumed, never verified.** The bandwidth and density checks
  can falsify the design; passing them does not establish it.
- **A sharp design only.** Where treatment at the threshold merely becomes more
  likely rather than certain, a fuzzy RDD is required, which identifies a LATE
  among compliers.
- **The bandwidth is a researcher choice**, and here a consequential one: the
  estimate moves by roughly a quarter of the effect across the grid. A single
  reported bandwidth hides that. Data-driven selectors exist and are not
  implemented in this project.
- **Synthetic data.** The discontinuity here was constructed. Real running
  variables are measured with error, which attenuates the estimate.